In [1]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [2]:
import pandas as pd
data=pd.read_csv("./data/demo.csv")
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### pipe构建

In [4]:
from easymlops import TablePipeLine
from easymlops.table.encoding import *
from easymlops.table.preprocessing import *
from easymlops.table.decomposition import *

In [5]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass","Sex"],drop_col=False))\
  .pipe(LabelEncoding(cols=["Sex","Pclass"]))\
  .pipe(TargetEncoding(cols=["Name","Ticket","Embarked","Cabin"],y=y_train))\
  .pipe(PCADecomposition(n_components=4))

x_test_new=table.fit(x_train).transform(x_test)
x_test_new.head(5)

,0,1,2,3
500,250.112431,-27.094711,-5.835574,-0.250618
501,251.117575,-27.778216,-1.808170,-0.265342
502,252.017783,-29.145885,-22.761863,-0.698056
503,253.218449,-25.038811,14.050332,0.200772
504,255.219670,50.502969,-11.396099,-1.079058


### 性能、一致性测试

In [ ]:
table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.3ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.OneHotEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.LabelEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.TargetEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.decomposition.tradition.PCADecomposition'>) module check [transform] complete,speed:[6.16ms]/it,cpu:[41%],memory:[0K]

#####################

### 持久化测试

In [7]:
table.save("ml.pkl")

In [8]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass","Sex"],drop_col=False))\
  .pipe(LabelEncoding(cols=["Sex","Pclass"]))\
  .pipe(TargetEncoding(cols=["Name","Ticket","Embarked","Cabin"]))\
  .pipe(PCADecomposition(n_components=4))

table.load("ml.pkl")

In [9]:
table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.3ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.OneHotEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.LabelEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.TargetEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.decomposition.tradition.PCADecomposition'>) module check [transform] complete,speed:[7.76ms]/it,cpu:[37%],memory:[20K]

####################

In [10]:
table.transform(x_test[:5])

,0,1,2,3
500,250.112431,-27.094711,-5.835574,-0.250618
501,251.117575,-27.778216,-1.808170,-0.265342
502,252.017783,-29.145885,-22.761863,-0.698056
503,253.218449,-25.038811,14.050332,0.200772
504,255.219670,50.502969,-11.396099,-1.079058


In [11]:
x_test_new.head(5)

,0,1,2,3
500,250.112431,-27.094711,-5.835574,-0.250618
501,251.117575,-27.778216,-1.808170,-0.265342
502,252.017783,-29.145885,-22.761863,-0.698056
503,253.218449,-25.038811,14.050332,0.200772
504,255.219670,50.502969,-11.396099,-1.079058
